In [ ]:
import pandas as pd
import re
import os

archivo_original = 'drugbank.csv'
archivo_temp = 'temp.csv'
archivo_resumen_txt = 'drugbank_diccionario_medico.txt' # Renombrado para claridad
archivo_csv_completo = 'drugbank_completo_estructurado.csv' # Nuevo nombre para el CSV completo

print("Paso 1: Reparando estructura original...")
lineas_reparadas = []
registro_actual = ""
patron_id = re.compile(r'^"?DB\d{5}')

with open(archivo_original, 'r', encoding='utf-8') as f:
    cabecera = f.readline()
    lineas_reparadas.append(cabecera)
    for linea in f:
        if patron_id.match(linea):
            if registro_actual:
                lineas_reparadas.append(registro_actual + '\n')
            registro_actual = linea.strip()
        else:
            registro_actual += " " + linea.strip()
    if registro_actual:
        lineas_reparadas.append(registro_actual + '\n')

with open(archivo_temp, 'w', encoding='utf-8') as f:
    f.writelines(lineas_reparadas)

print("Paso 2: Cargando datos y resumiendo...")
df = pd.read_csv(archivo_temp, sep=',', low_memory=False)

# --- Guardar el DataFrame completo en CSV ---
# Se reordena para que las columnas clave estén al inicio
nuevo_orden_csv = [
    'drugbank-id', 'name', 'description', 'indication', 'mechanism-of-action',
    'synonyms', 'cas-number', 'unii', 'state', 'groups', 'salts',
    'pharmacodynamics', 'toxicity', 'drug-interactions', 'food-interactions',
    'metabolism', 'absorption', 'half-life', 'protein-binding', 'route-of-elimination', 'volume-of-distribution', 'clearance',
    'classification', 'categories', 'atc-codes', 'ahfs-codes',
    'products', 'international-brands', 'mixtures', 'packagers', 'manufacturers', 'prices', 'dosages', 'fda-label',
    'targets', 'enzymes', 'carriers', 'transporters', 'affected-organisms', 'pathways', 'reactions', 'average-mass', 'monoisotopic-mass', 'calculated-properties',
    'general-references', 'synthesis-reference', 'pdb-entries', 'msds', 'patents', 'sequences', 'experimental-properties', 'external-identifiers', 'external-links', 'snp-effects', 'snp-adverse-drug-reactions'
]

# Conservamos solo las columnas que existan en el DataFrame y las reordenamos
columnas_para_csv = [col for col in nuevo_orden_csv if col in df.columns]
df_ordenado_csv = df[columnas_para_csv]

# Limpiar texto y NaNs para el CSV completo (similar a lo que ya tenías)
def limpiar_texto_csv(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = re.sub(r'[\n\r\t]', ' ', texto) # Borra los "Enters" ocultos
    texto = re.sub(r'<.*?>', '', texto)     # Borra código HTML
    texto = re.sub(r'\s+', ' ', texto)      # Unifica espacios
    texto = texto.replace('"', "'")         # Cambia comillas para no romper el CSV
    return texto.strip()

for columna in df_ordenado_csv.columns:
    if df_ordenado_csv[columna].dtype == object:
        df_ordenado_csv[columna] = df_ordenado_csv[columna].apply(limpiar_texto_csv)

# Rellenar NaNs de forma específica para evitar el FutureWarning
# Rellenar con "" EXCLUSIVAMENTE las columnas de texto que puedan tener NaNs restantes
columnas_texto_csv = df_ordenado_csv.select_dtypes(include=['object']).columns
df_ordenado_csv[columnas_texto_csv] = df_ordenado_csv[columnas_texto_csv].fillna("")

# Rellenar con 0 EXCLUSIVAMENTE las columnas numéricas que puedan tener NaNs restantes
columnas_numericas_csv = df_ordenado_csv.select_dtypes(include=['float64', 'int64']).columns
df_ordenado_csv[columnas_numericas_csv] = df_ordenado_csv[columnas_numericas_csv].fillna(0)

df_ordenado_csv.to_csv(archivo_csv_completo, index=False, encoding='utf-8', quoting=1)
print(f"El DataFrame completo y estructurado se ha guardado como: {archivo_csv_completo}")
# --- Fin de la sección CSV ---


# Elegimos solo lo que un humano necesita leer para entender el fármaco para el TXT
columnas_clave_txt = ['drugbank-id', 'name', 'description', 'indication', 'mechanism-of-action']
columnas_clave_txt = [col for col in columnas_clave_txt if col in df.columns] # Asegurar que existan

print("Paso 3: Generando el diccionario de texto limpio...")
with open(archivo_resumen_txt, 'w', encoding='utf-8') as f:

    # Recorremos cada medicamento uno por uno
    for index, row in df.iterrows(): # Usamos el df original antes de ordenar las columnas para el CSV
        # Escribimos el título principal del fármaco
        nombre = str(row['name']).strip() if pd.notna(row['name']) else "Desconocido"
        db_id = str(row['drugbank-id']).strip() if pd.notna(row['drugbank-id']) else "Sin ID"

        f.write(f"=== {nombre} ({db_id}) ===\n")

        # Procesamos solo las columnas de información para el TXT
        for col in columnas_clave_txt:
            if col in ['name', 'drugbank-id']:
                continue

            valor = row[col]

            # MAGIA AQUÍ: Solo escribimos si el dato NO es nulo y NO está vacío
            if pd.notna(valor) and str(valor).strip() != "":
                # Limpiamos el texto de basura HTML y espacios raros
                texto_limpio = re.sub(r'<.*?>', '', str(valor))
                texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()

                # Ponemos el nombre de la categoría en mayúsculas bonitas (ej. indication -> Indication)
                titulo_categoria = col.replace('-', ' ').title()
                f.write(f"• {titulo_categoria}: {texto_limpio}\n\n")

        # Separador visual entre medicamentos
        f.write("-" * 80 + "\n\n")

if os.path.exists(archivo_temp):
    os.remove(archivo_temp)

print(f"¡Listo! Descarga tu archivo de texto: {archivo_resumen_txt}")

Paso 1: Reparando estructura original...
Paso 2: Cargando datos y resumiendo...
El DataFrame completo y estructurado se ha guardado como: drugbank_completo_estructurado.csv
Paso 3: Generando el diccionario de texto limpio...
¡Listo! Descarga tu archivo de texto: drugbank_diccionario_medico.txt


In [ ]:
import pandas as pd
import re
import os

archivo_original = 'drugbank.csv'
archivo_reparado_temporal = 'drugbank_temp.csv'
archivo_final_csv = 'drugbank_personalizado.csv' # Nombre para el CSV
archivo_final_txt = 'drugbank_personalizado_resumen.txt' # Nombre para el TXT

# 1. REPARAR EL ARCHIVO COMPLETO
print("Paso 1: Reconstruyendo filas rotas en todo el documento...")
lineas_reparadas = []
registro_actual = ""
patron_id = re.compile(r'^"?DB\d{5}')

with open(archivo_original, 'r', encoding='utf-8') as f:
    cabecera = f.readline()
    lineas_reparadas.append(cabecera)

    for linea in f:
        if patron_id.match(linea):
            if registro_actual:
                lineas_reparadas.append(registro_actual + '\n')
            registro_actual = linea.strip()
        else:
            registro_actual += " " + linea.strip()

    if registro_actual:
        lineas_reparadas.append(registro_actual + '\n')

with open(archivo_reparado_temporal, 'w', encoding='utf-8') as f:
    f.writelines(lineas_reparadas)

# 2. CARGAR Y APLICAR TU SELECCIÓN DE COLUMNAS
print("Paso 2: Cargando datos y aplicando tu configuración exacta de columnas...")
df = pd.read_csv(archivo_reparado_temporal, sep=',', low_memory=False)

# Lista con tus instrucciones: Se conservan patentes y propiedades, se eliminan secuencias y referencias.
columnas_personalizadas = [
    'drugbank-id', 'name', 'synonyms', 'cas-number', 'unii', 'state', 'groups', 'salts',
    'description', 'indication', 'mechanism-of-action', 'pharmacodynamics', 'toxicity',
    'drug-interactions', 'food-interactions',
    'metabolism', 'absorption', 'half-life', 'protein-binding', 'route-of-elimination', 'volume-of-distribution', 'clearance',
    'classification', 'categories', 'atc-codes', 'ahfs-codes',
    'products', 'international-brands', 'mixtures', 'packagers', 'manufacturers', 'prices', 'dosages', 'fda-label',
    'targets', 'enzymes', 'carriers', 'transporters', 'affected-organisms', 'pathways', 'reactions', 'average-mass', 'monoisotopic-mass',
    'calculated-properties', 'experimental-properties', 'pdb-entries', 'msds', 'patents',
    'external-identifiers', 'external-links', 'snp-effects', 'snp-adverse-drug-reactions'
]

# Retenemos únicamente estas 52 columnas
columnas_existentes = [col for col in columnas_personalizadas if col in df.columns]
df = df[columnas_existentes]

# 3. LIMPIAR TEXTO Y MANEJAR NaNs CORRECTAMENTE
print("Paso 3: Blindando formatos y rellenando celdas vacías (sin advertencias)...")
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    # Eliminar patrones de identificación de bases de datos y secuencias al principio o en el texto
    # Ejemplo: Database31697UniProtKBQ99062UniProt AccessionCSF3R_HUMANG-CSF receptorGCSFR
    # Este regex es más general para cubrir varias combinaciones de identificadores.
    texto = re.sub(r'Database\d+UniProtKB[A-Z0-9]+UniProt Accession[^>]*', '', texto, flags=re.IGNORECASE)
    # Eliminar bloques de secuencias que comienzan con '>lcl|BSEQ' y su contenido
    # Captura todo desde '>lcl|BSEQ' hasta el siguiente '>' o el final del string.
    texto = re.sub(r'>lcl\|BSEQ\d+\|[^>]*', '', texto)

    # Limpieza general ya existente
    texto = re.sub(r'[\n\r\t]', ' ', texto) # Elimina saltos de línea ocultos
    texto = re.sub(r'<.*?>', '', texto)     # Elimina código HTML
    texto = re.sub(r'\s+', ' ', texto)      # Unifica espacios
    texto = texto.replace('"', "'")         # Cambia comillas para proteger el CSV
    return texto.strip()

# Seleccionar solo las columnas que son de texto para evitar el warning numérico
columnas_texto = df.select_dtypes(include=['object']).columns

for columna in columnas_texto:
    df[columna] = df[columna].apply(limpiar_texto)

# Rellenar con "" SOLO las columnas de texto
df[columnas_texto] = df[columnas_texto].fillna("")

# Rellenar con 0 SOLO las columnas numéricas
columnas_numericas = df.select_dtypes(include=['float64', 'int64']).columns
df[columnas_numericas] = df[columnas_numericas].fillna(0)

# 4. GUARDAR EL CSV FINAL
print("Paso 4: Exportando tu CSV personalizado...")
df.to_csv(archivo_final_csv, index=False, encoding='utf-8', quoting=1)
print(f"Tu archivo CSV final es: {archivo_final_csv}")

# --- Sección para generar el archivo de texto resumen ---
columnas_clave_para_txt = ['drugbank-id', 'name', 'description', 'indication', 'mechanism-of-action']
columnas_clave_para_txt = [col for col in columnas_clave_para_txt if col in df.columns] # Asegurar que existan

print("Paso 5: Generando un archivo de texto resumen (TXT)...")
with open(archivo_final_txt, 'w', encoding='utf-8') as f:
    for index, row in df.iterrows():
        nombre = str(row['name']).strip() if pd.notna(row['name']) else "Desconocido"
        db_id = str(row['drugbank-id']).strip() if pd.notna(row['drugbank-id']) else "Sin ID"

        f.write(f"=== {nombre} ({db_id}) ===\n")

        for col in columnas_clave_para_txt:
            if col in ['name', 'drugbank-id']:
                continue

            valor = row[col]
            if pd.notna(valor) and str(valor).strip() != "": # Se usa pd.notna(valor) y luego str(valor).strip() != "" para cubrir todos los casos de "vacío"
                texto_limpio = re.sub(r'<.*?>', '', str(valor))
                texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()
                titulo_categoria = col.replace('-', ' ').title()
                f.write(f"• {titulo_categoria}: {texto_limpio}\n\n")
        f.write("-" * 80 + "\n\n")
print(f"Tu archivo de texto resumen se ha guardado como: {archivo_final_txt}")
# --- Fin de la sección de texto resumen ---

# Limpieza del entorno
if os.path.exists(archivo_reparado_temporal):
    os.remove(archivo_reparado_temporal)

print(f"\n¡Proceso terminado exitosamente! Puedes encontrar tus archivos en {os.getcwd()}")
print(f"CSV: {archivo_final_csv}")
print(f"TXT: {archivo_final_txt}")

Paso 1: Reconstruyendo filas rotas en todo el documento...
Paso 2: Cargando datos y aplicando tu configuración exacta de columnas...
Paso 3: Blindando formatos y rellenando celdas vacías (sin advertencias)...
Paso 4: Exportando tu CSV personalizado...
Tu archivo CSV final es: drugbank_personalizado.csv
Paso 5: Generando un archivo de texto resumen (TXT)...
Tu archivo de texto resumen se ha guardado como: drugbank_personalizado_resumen.txt

¡Proceso terminado exitosamente! Puedes encontrar tus archivos en /content
CSV: drugbank_personalizado.csv
TXT: drugbank_personalizado_resumen.txt


Datos Prescindibles (Candidatos a Eliminar)
Estas columnas contienen texto en bruto, cadenas de ADN o metadatos de laboratorio que inflan el peso del archivo. Solo aportan valor en investigaciones de química computacional profunda y generan lentitud al iterar bases de datos convencionales.

sequences: Cadenas de cientos o miles de caracteres genéticos que causan desbordamientos de lectura.(ELIMINAR)

general-references y synthesis-reference: Bloques inmensos de bibliografía médica y citas crudas.(ELIMINAR)

patents y msds: Metadatos de propiedad intelectual y hojas de seguridad industrial. (DEJARLO)

calculated-properties, experimental-properties, pdb-entries: Estructuras moleculares 3D, pesos isotópicos y fórmulas complejas que saturan el almacenamiento.(DEJARLO)

Al descartar este último bloque, la base de datos reducirá drásticamente su peso en megabytes y aumentará la velocidad de respuesta, manteniendo intacto el valor médico.

In [ ]:
import pandas as pd
import re
import os

archivo_original = 'drugbank.csv'
archivo_temp = 'temp_reparado.csv'
archivo_resumen_completo = 'drugbank_fichas_completas.txt'

print("Paso 1: Reparando estructura original del archivo...")
lineas_reparadas = []
registro_actual = ""
patron_id = re.compile(r'^"?DB\d{5}')

with open(archivo_original, 'r', encoding='utf-8') as f:
    cabecera = f.readline()
    lineas_reparadas.append(cabecera)
    for linea in f:
        if patron_id.match(linea):
            if registro_actual:
                lineas_reparadas.append(registro_actual + '\n')
            registro_actual = linea.strip()
        else:
            registro_actual += " " + linea.strip()
    if registro_actual:
        lineas_reparadas.append(registro_actual + '\n')

with open(archivo_temp, 'w', encoding='utf-8') as f:
    f.writelines(lineas_reparadas)

print("Paso 2: Cargando datos...")
df = pd.read_csv(archivo_temp, sep=',', low_memory=False)

# Columnas excluidas exactamente como lo pediste
columnas_excluidas = ['sequences', 'general-references', 'synthesis-reference',
                      'targets', 'experimental-properties', 'external-identifiers',
                      'external-links', 'packagers', 'products']
# Mantenemos todas las demás columnas, dejando ID y Nombre como la cabecera principal de la ficha
columnas_a_mostrar = [col for col in df.columns if col not in columnas_excluidas and col not in ['drugbank-id', 'name']]

print("Paso 3: Generando el documento de texto detallado con todo lo demás...")
with open(archivo_resumen_completo, 'w', encoding='utf-8') as f:

    for index, row in df.iterrows():
        nombre = str(row['name']).strip() if pd.notna(row['name']) else "Desconocido"
        db_id = str(row['drugbank-id']).strip() if pd.notna(row['drugbank-id']) else "Sin ID"

        f.write(f"=== {nombre} ({db_id}) ===\n")

        # Recorremos cada una de las columnas permitidas
        for col in columnas_a_mostrar:
            valor = row[col]

            # Solo escribimos si el campo tiene información real y útil
            if pd.notna(valor) and str(valor).strip() != "" and str(valor).strip() != "Sin registro":
                # Limpiamos etiquetas HTML y espacios múltiples para una lectura fluida
                texto_limpio = re.sub(r'<.*?>', '', str(valor))
                texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()

                # Formateamos el título de la categoría
                titulo_categoria = col.replace('-', ' ').title()
                f.write(f"• {titulo_categoria}: {texto_limpio}\n\n")

        f.write("=" * 90 + "\n\n")

if os.path.exists(archivo_temp):
    os.remove(archivo_temp)

print(f"\n¡Listo! Descarga tu archivo de texto con todo el contenido ordenado: {archivo_resumen_completo}")

Paso 1: Reparando estructura original del archivo...
Paso 2: Cargando datos...
Paso 3: Generando el documento de texto detallado con todo lo demás...

¡Listo! Descarga tu archivo de texto con todo el contenido ordenado: drugbank_fichas_completas.txt
